# Phân tích và tiền xử lý dữ liệu giá nhà (House Prices)

Notebook này thực hiện các bước chuẩn bị dữ liệu cho bài toán **dự đoán giá bán nhà (`SalePrice`)**:

1. Nạp dữ liệu và khám phá tổng quan
2. Phân tích dữ liệu thiếu
3. Phân tích biến mục tiêu `SalePrice`
4. Phân tích mối quan hệ giữa diện tích (`GrLivArea`) và giá
5. Làm sạch dữ liệu: loại outlier, xử lý giá trị thiếu
6. Chia dữ liệu thành tập Train / Validation / Test

> **Lưu ý:** cần đặt file `train.csv` cùng thư mục với notebook (hoặc ở `/content/train (1).csv` nếu chạy trên Colab).

## 1. Khai báo thư viện và cấu hình chung

- `numpy`, `pandas`: xử lý dữ liệu dạng bảng.
- `matplotlib`, `seaborn`: vẽ biểu đồ.
- `SimpleImputer`: điền giá trị thiếu theo một chiến lược (median, most_frequent...).
- `train_test_split`: chia dữ liệu thành các tập con.
- `RANDOM_STATE = 42`: cố định hạt giống ngẫu nhiên để kết quả chia dữ liệu **lặp lại được** ở mỗi lần chạy.
- Hàm `show()`: dùng `display()` khi chạy trong Jupyter/Colab (bảng đẹp hơn), nếu không có thì rơi về `print()`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42


def show(obj):
    try:
        display(obj)
    except NameError:
        print(obj)

## 2. Nạp dữ liệu

Đoạn code tìm file `train.csv` trong danh sách đường dẫn ứng viên (`CANDIDATE_PATHS`). Đường dẫn đầu tiên tồn tại sẽ được dùng; nếu không có đường dẫn nào hợp lệ thì báo lỗi `FileNotFoundError` kèm hướng dẫn.

Sau khi đọc bằng `pd.read_csv`, ta in số dòng, số cột và xem 5 dòng đầu (`head()`) để kiểm tra nhanh dữ liệu đã nạp đúng chưa.

In [ ]:
CANDIDATE_PATHS = [
    "train.csv",
    "/content/train (1).csv",
]
data_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(
        "Không tìm thấy train.csv. Hãy đặt file cùng thư mục với notebook "
        "hoặc sửa lại CANDIDATE_PATHS."
    )

df = pd.read_csv(data_path)

print("Nạp dữ liệu thành công!")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
show(df.head())

## 3. Khám phá tổng quan dữ liệu

- `df.info()`: kiểu dữ liệu từng cột và số giá trị không rỗng → nhìn nhanh được cột nào thiếu dữ liệu.
- `df.describe()`: thống kê mô tả (mean, std, min, quartile, max) cho các thuộc tính **số**.
- `df.describe(include="object")`: thống kê cho các thuộc tính **phân loại** (số giá trị khác nhau, giá trị xuất hiện nhiều nhất, tần suất).
- `df.duplicated().sum()`: đếm số dòng trùng lặp; nếu bằng 0 thì không cần khử trùng.

In [ ]:
df.info()

print("\nThống kê các thuộc tính số:")
show(df.describe())

print("\nThống kê các thuộc tính phân loại:")
show(df.describe(include="object"))

print("\nSố dòng trùng lặp:", df.duplicated().sum())

## 4. Phân tích dữ liệu thiếu

Đếm số giá trị `NaN` của từng cột, chỉ giữ các cột có thiếu và sắp xếp giảm dần, sau đó vẽ biểu đồ cột.

Cách đọc kết quả: cột thiếu nhiều (ví dụ `PoolQC`, `MiscFeature`, `Alley`, `Fence`) thường **không phải do lỗi nhập liệu** mà vì ngôi nhà đơn giản *không có* tiện ích đó (không có hồ bơi, không có hẻm...). Điều này quyết định cách xử lý ở bước 7.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Các cột bị thiếu dữ liệu:")
show(missing.to_frame(name="Số lượng thiếu"))

plt.figure(figsize=(12, 6))
sns.barplot(x=missing.index, y=missing.values)
plt.title("Số lượng giá trị thiếu theo từng thuộc tính")
plt.xlabel("Thuộc tính")
plt.ylabel("Số lượng giá trị thiếu")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 5. Phân tích biến mục tiêu `SalePrice`

- **Skewness (độ lệch)**: > 0 nghĩa là phân phối lệch phải (đuôi dài về phía giá cao). Giá nhà thường lệch phải vì có một số ít căn rất đắt.
- **Kurtosis (độ nhọn)**: cho biết mức độ "đuôi dày" so với phân phối chuẩn (chuẩn có kurtosis = 0 theo định nghĩa của pandas).
- **Histogram + KDE**: quan sát hình dạng phân phối.
- **Boxplot**: các điểm nằm ngoài "râu" là giá trị ngoại lai tiềm năng.

Nếu phân phối lệch mạnh, khi huấn luyện mô hình hồi quy ta thường cân nhắc biến đổi log (`np.log1p`) cho `SalePrice`.

In [ ]:
print("Skewness của SalePrice:", round(df["SalePrice"].skew(), 2))
print("Kurtosis của SalePrice:", round(df["SalePrice"].kurt(), 2))

plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="SalePrice", kde=True)
plt.title("Phân phối giá bán nhà")
plt.xlabel("SalePrice")
plt.ylabel("Tần suất")
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(x=df["SalePrice"])
plt.title("Boxplot giá bán nhà")
plt.xlabel("SalePrice")
plt.show()

## 6. Mối quan hệ giữa diện tích sinh hoạt và giá bán

Tính hệ số tương quan Pearson giữa `GrLivArea` (diện tích sinh hoạt trên mặt đất) và `SalePrice`, đồng thời vẽ biểu đồ phân tán (scatter plot).

Hệ số gần +1 nghĩa là hai biến tương quan thuận mạnh. Trên scatter plot, ta cũng tìm các điểm **bất thường**: diện tích rất lớn nhưng giá lại thấp – đây là cơ sở để loại outlier ở bước tiếp theo.

In [ ]:
print(
    "Tương quan GrLivArea - SalePrice:",
    round(df["GrLivArea"].corr(df["SalePrice"]), 3),
)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice")
plt.title("Mối quan hệ giữa diện tích và giá nhà")
plt.xlabel("Diện tích sinh hoạt (GrLivArea)")
plt.ylabel("Giá bán (SalePrice)")
plt.show()

## 7. Làm sạch dữ liệu

Tạo bản sao `df_clean` để giữ nguyên dữ liệu gốc `df`.

### 7.1. Loại bỏ outlier

Giữ lại các căn có `GrLivArea <= 4000`. Những căn diện tích rất lớn (> 4000) nhưng giá thấp bất thường là các điểm nhiễu, có thể làm lệch đường hồi quy nếu giữ lại. Code in ra các quan sát bị loại (kèm `SaleCondition`) để ta kiểm tra trước khi xóa.

In [ ]:
df_clean = df.copy()

outliers = df_clean[df_clean["GrLivArea"] > 4000]
print("Số quan sát GrLivArea > 4000:", len(outliers))
show(outliers[["Id", "GrLivArea", "SalePrice", "SaleCondition"]])

df_clean = df_clean[df_clean["GrLivArea"] <= 4000].copy()
print("Kích thước sau khi loại outlier:", df_clean.shape)

### 7.2. Xử lý giá trị thiếu "có ý nghĩa"

Với nhiều cột, `NaN` có nghĩa là **"không có"** chứ không phải dữ liệu bị mất:

- **Cột phân loại** (`categorical_none_cols`): ví dụ không có tầng hầm → `BsmtQual` là NaN, không có gara → `GarageType` là NaN. Ta điền chuỗi `"None"` để coi đây là một nhóm riêng.
- **Cột số** (`numeric_zero_cols`): ví dụ không có tầng hầm thì `TotalBsmtSF` = 0, không có gara thì `GarageArea` = 0. Ta điền `0`.

Làm bước này **trước** khi dùng imputer thống kê để tránh điền nhầm giá trị phổ biến cho những căn thực sự không có tiện ích.

In [ ]:
categorical_none_cols = [
    "Alley", "MasVnrType", "BsmtQual", "BsmtCond",
    "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu", "GarageType", "GarageFinish",
    "GarageQual", "GarageCond", "PoolQC",
    "Fence", "MiscFeature",
]
numeric_zero_cols = [
    "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2",
    "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath",
    "BsmtHalfBath", "GarageCars", "GarageArea",
]

print("Trước khi xử lý thiếu:")
print("Số cột thiếu:", df_clean.isnull().any().sum())
print("Số ô thiếu:", df_clean.isnull().sum().sum())

df_clean[categorical_none_cols] = df_clean[categorical_none_cols].fillna("None")
df_clean[numeric_zero_cols] = df_clean[numeric_zero_cols].fillna(0)

### 7.3. Xử lý giá trị thiếu "thật sự"

Sau bước 7.2, các ô còn thiếu là thiếu thật (ví dụ `LotFrontage`, `GarageYrBlt`, `Electrical`). Cách làm:

1. Tách danh sách cột đặc trưng (`feature_cols`), **bỏ `Id` và `SalePrice`** (`Id` chỉ là mã định danh, `SalePrice` là nhãn).
2. Chia thành cột số (`numeric_cols`) và cột phân loại (`categorical_cols`).
3. Điền **median (trung vị)** cho cột số – ít bị ảnh hưởng bởi giá trị cực đoan hơn mean.
4. Điền **most_frequent (giá trị xuất hiện nhiều nhất)** cho cột phân loại.

> **Điểm cần lưu ý:** ở đây imputer được `fit` trên toàn bộ `df_clean` *trước khi* chia Train/Val/Test, nên median/mode được tính cả từ dữ liệu Val/Test (rò rỉ thông tin nhỏ – *data leakage*). Với bài tập thì chấp nhận được; nếu muốn chặt chẽ hơn, hãy chia dữ liệu trước rồi chỉ `fit` imputer trên tập Train (ví dụ dùng `Pipeline`).

In [ ]:
feature_cols = [c for c in df_clean.columns if c not in ("Id", "SalePrice")]
numeric_cols = (
    df_clean[feature_cols].select_dtypes(include=np.number).columns.tolist()
)
categorical_cols = (
    df_clean[feature_cols].select_dtypes(exclude=np.number).columns.tolist()
)

print("Các cột còn thiếu (thiếu thật sự):")
still_missing = df_clean[feature_cols].isnull().sum()
show(still_missing[still_missing > 0].to_frame("Số lượng thiếu"))

numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

df_clean[numeric_cols] = numeric_imputer.fit_transform(df_clean[numeric_cols])
df_clean[categorical_cols] = categorical_imputer.fit_transform(
    df_clean[categorical_cols]
)

print("\nSau khi xử lý thiếu:")
print("Số ô thiếu trong df_clean:", df_clean.isnull().sum().sum())

## 8. Chia dữ liệu Train / Validation / Test

Tách đặc trưng `X` (bỏ `SalePrice`, `Id`) và nhãn `y` (`SalePrice`), rồi chia hai lần:

1. Lần 1: **70% Train** và 30% còn lại (`X_temp`).
2. Lần 2: chia đôi 30% đó → **15% Validation** và **15% Test**.

Vai trò của từng tập:
- **Train**: huấn luyện mô hình.
- **Validation**: chọn siêu tham số, so sánh các mô hình.
- **Test**: đánh giá cuối cùng, chỉ dùng **một lần** để có ước lượng khách quan.

Các dòng cuối in kích thước và tỷ lệ thực tế để xác nhận việc chia đúng 70/15/15.

In [ ]:
X = df_clean.drop(columns=["SalePrice", "Id"])
y = df_clean["SalePrice"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print("Kích thước tập Train:     ", X_train.shape, y_train.shape)
print("Kích thước tập Validation:", X_val.shape, y_val.shape)
print("Kích thước tập Test:      ", X_test.shape, y_test.shape)

print("\nTỷ lệ phân chia:")
print(f"Train:      {len(X_train) / len(X):.1%}")
print(f"Validation: {len(X_val) / len(X):.1%}")
print(f"Test:       {len(X_test) / len(X):.1%}")